# Step 1.3 — Create a dedicated metadata schema

Use your existing catalog:
* *dev_catalog*.hrs_agent_metadata

This is a proposed schema for the AI Agent, separate from slv_cdm_hrs and gld_star_hrs. Create it only after confirming that the name is available and that you have the necessary permissions.

Start with these metadata tables:

|Table 	|Essential columns|
| --------- | --------------- |
|hrs_variable_catalog|	variable_name, variable_label, definition, survey_section, source_page, codebook_version|
|hrs_variable_wave_coverage|	variable_name, wave_number, survey_year, coverage_status, source_page|
|hrs_variable_cohort_coverage|	variable_name, hacohort_number, coverage_status, source_page|
|hrs_variable_statistics|	variable_name, wave_number, hacohort_number, statistic_name, statistic_value, source_page|
|hrs_variable_value_labels|	variable_name, value_code, value_label, source_page|
|hrs_document_chunks|	document_id, page_number, section_heading, chunk_text, codebook_version |

These are proposed starting schemas. Refine the keys and column types after inspecting the actual Codebook layout. Use DOUBLE for numeric statistical values when appropriate, and preserve categorical codes without assuming they are continuous numeric measures.

The variable catalog should distinguish the source variable name from the warehouse column name. For example, a source variable such as R1BMI should not be confused with a derived Gold measure such as average_bmi.

# What Each Table Does
## Table 1 — hrs_variable_catalog

* Question it answers: What does an HRS variable mean?

This is the master catalog of variables. Each variable has a definition and supporting source information from the Codebook.

|variable_name	|variable_label	|survey_section	|source_page|
| ------------- | ------------- | ------------- | --------- |
|R1BMI	|Example BMI variable label	|Health	|Example: 120|

Illustrative example only; the label and page must be verified against the actual Codebook.

The full table would also include fields such as definition and codebook_version.

Why the agent needs it:
- Explains what a variable measures.
- Identifies the relevant HRS subject area, such as Demographics or Health.
- Helps select source variables for a business question.
- Provides traceability to the Codebook.

This is the central reference table for the other variable-related tables.

## Table 2 — hrs_variable_wave_coverage

* Question it answers: In which HRS waves is a variable available?

The RAND HRS Longitudinal dataset spans multiple survey waves. A variable might be available in one wave, several waves, or only during a particular period.

|variable_name	|wave_number	|survey_year	|coverage_status
| --- | --- | --- | --- |
|R1BMI	|1	|Example year	|Available|
|R2BMI	|2	|Example year	|Available|
|R3BMI	|3	|Example year	|Verify|

Illustrative rows, not verified Codebook coverage.

Why the agent needs it:
- Determines whether a variable can support a longitudinal analysis.
- Helps identify the correct wave-specific source column.
- Prevents the agent from assuming a variable exists in every wave.
- Helps generate SQL and DML that account for missing or changing variables.
The intended logical key is usually (variable_name, wave_number, codebook_version), assuming each combination has one coverage record.

# Table 3 — hrs_variable_cohort_coverage

* Question it answers: Which HRS cohorts does a variable apply to?

This is the table you asked about specifically, and it is important to distinguish a cohort from a wave.
- A cohort identifies a group of participants entering the study under a particular sampling design or recruitment period.
- A wave identifies a survey round or measurement occasion.

A variable may be available in a particular wave but only apply to some cohorts, depending on the survey design, eligibility rules, and when participants entered the study.

For example, a variable might be collected from a cohort beginning in an earlier period, while another cohort enters the study later. The exact coverage must be confirmed from the Codebook.

Why the agent needs it:
- Helps determine which respondents are eligible for an analysis.
- Identifies possible structural gaps in longitudinal data.
- Helps avoid treating a variable's absence as a respondent's missing answer when it was never collected for that cohort.
- Supports cohort-specific filtering and validation.

Example

|variable_name	|hacohort_number	|coverage_status	|source_page|
| --- | --- | --- | --- |
|R1BMI	|1	|status	|120|
|R1BMI	|2	|status	|120|
|R2BMI	|1	|status	|120|

However, there is a caveat: do not create this table just because the warehouse has a cohort dimension. 

Create it only if the Codebook documents meaningful variable-level cohort applicability that the agent needs to query. 

If cohort applicability is fully determined by wave coverage or the source dataset, a separate table may be unnecessary.

# Table 4 — hrs_variable_statistics

* Question it answers: What descriptive statistics does the Codebook report for a variable?

The Codebook can report statistics such as minimum, maximum, mean, standard deviation, or the number of observations. The actual statistics available depend on the variable and Codebook tables.

|variable_name	|wave_number	|statistic_name	|statistic_value|
| --- | --- | --- | --- |
|R1BMI	|1	|Minimum	|Illustrative|
|R1BMI	|1	|Maximum	|Illustrative|
|R1BMI	|1	|Mean	    |Illustrative|

Why the agent needs it:
- Helps determine suitable SQL data types.
- Provides reference values for data-quality checks.
- Helps detect unexpected values after ingestion.
- Supports validation of the Silver layer against the Codebook.

I would retain hacohort_number only if the Codebook's statistics are actually reported by cohort. If the statistics are reported only by wave, that column should be nullable or omitted from that particular design. Also consider statistic_unit or sample_size if those details are available.

# Table 5 — hrs_variable_value_labels

* Question it answers: What does each coded value mean?

Survey data frequently uses numeric codes to represent categories, rather than storing descriptive text in every record.

For example, a hypothetical categorical variable might have labels like these:

|variable_name	|value_code	|value_label|
| --- | --- | --- |
|Example variable	|1	|Category A|
|Example variable	|2	|Category B|
|Example variable	|8	|Don't know|

Illustrative labels only.

Why the agent needs it:
- Converts coded values into understandable categories.
- Helps generate meaningful SQL CASE expressions.
- Supports race, gender, health-response, and other categorical analyses.
- Helps distinguish valid responses from special missing-value codes.

The key should include variable_name, value_code, and potentially wave_number or codebook_version if the meaning of a code changes across waves or Codebook editions.

# Table 6 — hrs_document_chunks

* Question it answers: Which passages of the Codebook provide context for a question?

Your Step 1.2 notebook extracts text page by page. A chunk is a smaller passage of text—possibly a paragraph, a definition, or a section of a table—prepared for searching by meaning.

For example, a chunk might contain a variable definition, instructions about how to interpret a value, or a note describing changes between waves.

Why the agent needs it:
- Retrieves relevant Codebook passages when a user asks a question in natural language.
- Provides context that may not fit neatly into a structured variable record.
- Supports semantic search and retrieval-augmented generation (RAG).
- Gives the agent source text to cite or inspect when explaining a recommendation.

This table is especially useful when building an AI Agent. It complements the structured tables; it does not replace them.

# Recommended approach
For your six metadata tables, I recommend:
- Keep the logical composite keys documented.  No need to create constraints for them. Enforce with DML.
- Declare primary keys where supported and useful for metadata clarity.
- Use validation queries to detect duplicate composite keys.
- Enforce uniqueness operationally in the DML process before inserting or updating records.

# Composite keys to validate
|Table	|Columns that should be unique together|
| --- | --- |
|hrs_variable_catalog	|variable_name, codebook_version|
|hrs_variable_wave_coverage	|variable_id, wave_number, codebook_version|
|hrs_variable_cohort_coverage	|variable_id, hacohort_number, codebook_version|
|hrs_variable_statistics	|variable_id, wave_number, statistic_name, codebook_version|
|hrs_variable_value_labels	|variable_id, value_code, applicable wave, codebook_version|

hrs_document_chunks	chunk_id is the generated identifier; additionally validate the chosen natural chunk identifier if you need repeatable ingestion

# DDL Statements
Target catalog: dev_catalog
Target schema: hrs_agent_metadata
Storage format: Delta
SQL dialect: Databricks SQL / Spark SQL

1. Create metadata schema

CREATE SCHEMA IF NOT EXISTS dev_catalog.hrs_agent_metadata;

2. hrs_variable_catalog

Purpose: Stores master definitions and Codebook references for RAND HRS variables.

CREATE TABLE IF NOT EXISTS

  dev_catalog.hrs_agent_metadata.hrs_variable_catalog (

    variable_id BIGINT GENERATED ALWAYS AS IDENTITY
      COMMENT 'Surrogate identifier for a variable definition',
    variable_name STRING NOT NULL
      COMMENT 'RAND HRS source variable name',
    variable_label STRING
      COMMENT 'Variable label documented in the Codebook',
    definition STRING
      COMMENT 'Description of the variable and what it measures',
    survey_section STRING
      COMMENT 'Codebook subject area or survey section',
    measurement_unit STRING
      COMMENT 'Measurement unit when documented',
    source_page INT
      COMMENT 'Codebook page supporting the variable definition',
    codebook_version STRING NOT NULL
      COMMENT 'Version of the Codebook used for extraction',
    created_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
      COMMENT 'Record creation timestamp',
    updated_timestamp TIMESTAMP
      COMMENT 'Timestamp of the last metadata update'
  )
USING DELTA;

Logical unique key: variable_name + codebook_version

3. hrs_variable_wave_coverage

Purpose: Records documented variable availability by survey wave.

CREATE TABLE IF NOT EXISTS

  dev_catalog.hrs_agent_metadata.hrs_variable_wave_coverage (

    variable_id BIGINT NOT NULL
      COMMENT 'Reference to the variable catalog',
    wave_number INT NOT NULL
      COMMENT 'HRS survey wave number',
    survey_year INT
      COMMENT 'Survey year associated with the wave',
    coverage_status STRING
      COMMENT 'Documented availability or applicability status',
    source_page INT
      COMMENT 'Codebook page supporting the coverage information',
    codebook_version STRING NOT NULL
      COMMENT 'Version of the Codebook used for extraction',
    created_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
      COMMENT 'Record creation timestamp'
  )
USING DELTA;

Logical unique key: variable_id + wave_number + codebook_version

4. hrs_variable_cohort_coverage

Purpose: Records documented variable applicability by HRS cohort. Populate only when cohort-level coverage is supported by the Codebook or approved documentation.

CREATE TABLE IF NOT EXISTS

  dev_catalog.hrs_agent_metadata.hrs_variable_cohort_coverage (

    variable_id BIGINT NOT NULL
      COMMENT 'Reference to the variable catalog',
    hacohort_number INT NOT NULL
      COMMENT 'HRS cohort identifier',
    coverage_status STRING
      COMMENT 'Documented availability or applicability status',
    source_page INT
      COMMENT 'Codebook page supporting the coverage information',
    codebook_version STRING NOT NULL
      COMMENT 'Version of the Codebook used for extraction',
    created_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
      COMMENT 'Record creation timestamp'
  )
USING DELTA;

Logical unique key: variable_id + hacohort_number + codebook_version

5. hrs_variable_statistics

Purpose: Stores Codebook descriptive statistics by wave only.

CREATE TABLE IF NOT EXISTS

  dev_catalog.hrs_agent_metadata.hrs_variable_statistics (

    variable_id BIGINT NOT NULL
      COMMENT 'Reference to the variable catalog',
    wave_number INT NOT NULL
      COMMENT 'HRS survey wave number',
    statistic_name STRING NOT NULL
      COMMENT 'Statistic type, such as minimum, maximum, mean, or standard deviation',
    statistic_value DOUBLE
      COMMENT 'Numeric statistic reported in the Codebook',
    source_page INT
      COMMENT 'Codebook page containing the statistic',
    codebook_version STRING NOT NULL
      COMMENT 'Version of the Codebook used for extraction',
    created_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
      COMMENT 'Record creation timestamp'
  )
USING DELTA;

Logical unique key: variable_id + wave_number + statistic_name + codebook_version

No cohort-level statistics or sample-size column is included because the Codebook statistics are reported by wave only and sample size is not confirmed as available.

6. hrs_variable_value_labels

Purpose: Stores meanings for coded values, including documented special response and missing-value codes.

CREATE TABLE IF NOT EXISTS

  dev_catalog.hrs_agent_metadata.hrs_variable_value_labels (


    variable_id BIGINT NOT NULL
      COMMENT 'Reference to the variable catalog',
    value_code STRING NOT NULL
      COMMENT 'Source value code, stored as text to preserve representation',
    value_label STRING
      COMMENT 'Meaning of the coded value',
    wave_number INT
      COMMENT 'Wave when the label is wave-specific; NULL otherwise',
    source_page INT
      COMMENT 'Codebook page containing the value label',
    codebook_version STRING NOT NULL
      COMMENT 'Version of the Codebook used for extraction',
    created_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
      COMMENT 'Record creation timestamp'
  )
USING DELTA;

Logical unique key: variable_id + value_code + applicable wave + codebook_version

If a variable's value labels change by wave, retain the wave-specific distinctions. Validate duplicates using a consistent rule for handling NULL wave values.

7. hrs_document_chunks

Purpose: Stores searchable Codebook text passages for AI Agent retrieval and source-grounded responses.

CREATE TABLE IF NOT EXISTS

  dev_catalog.hrs_agent_metadata.hrs_document_chunks (

    chunk_id BIGINT GENERATED ALWAYS AS IDENTITY
      COMMENT 'Surrogate identifier for the document chunk',
    document_id STRING NOT NULL
      COMMENT 'Stable identifier for the source document',
    page_number INT NOT NULL
      COMMENT 'One-based PDF page number',
    chunk_index INT
      COMMENT 'Sequence number of the chunk within the document or page',
    section_heading STRING
      COMMENT 'Codebook section or subsection heading',
    chunk_text STRING NOT NULL
      COMMENT 'Extracted text passage for search and retrieval',
    codebook_version STRING NOT NULL
      COMMENT 'Version of the Codebook used for extraction',
    created_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
      COMMENT 'Record creation timestamp'
  )
USING DELTA;

Identifier: chunk_id

The existing hrs_document_pages table remains the page-level extraction archive. This table stores smaller text passages prepared for search and retrieval.

8. Implementation notes

Execute the schema statement first, followed by the six table statements.

The DDL creates empty tables; separate DML or Python extraction code must populate them.

The related variable tables use variable_id rather than repeating variable_name.

Databricks constraints should not be assumed to enforce logical uniqueness or referential integrity. Add explicit duplicate and orphan-reference checks to your validation process.

CREATE TABLE IF NOT EXISTS does not modify an existing table's schema. Inspect existing tables before rerunning the script if their definitions differ.

The identity columns are system-generated. Do not assume identity values remain the same after a table is rebuilt and reloaded.

The cohort coverage table should be populated only when there is verified cohort-specific applicability information.

Next step: Implement the Step 1.3 parsing process to populate these metadata tables from hrs_document_pages, with source-page traceability and validation against the RAND HRS Codebook.
